# AI CV Screening — End-to-End POC

**Pipeline:** PDF/DOCX → text extraction → CV parsing → job requirement parsing → Sentence Transformers semantic similarity → hybrid scoring → candidate ranking → explanation.

> The POC uses a lightweight, transparent parser for reproducibility. In production, the parsing layer can be replaced or augmented with a dedicated NER model or LLM structured extraction.

In [ ]:
from pathlib import Path
import pandas as pd
from src.extractor import extract_text, parse_cv
from src.matcher import rank_candidates

JD_PATH = Path("../data/job_description.txt")
CV_DIR = Path("../data/sample_cvs")

jd = JD_PATH.read_text(encoding="utf-8")
jd

In [ ]:
profiles = []
for path in sorted(CV_DIR.iterdir()):
    text = extract_text(str(path))
    profile = parse_cv(text, str(path))
    profiles.append(profile)

[(p["name"], p["skills"], p["experience_years"], p["education"]) for p in profiles]

In [ ]:
results, job = rank_candidates(profiles, jd)

rows = [{
    "Rank": i,
    "Candidate": r["name"],
    "Overall": r["overall_score"],
    "Semantic": r["semantic_similarity"],
    "Skills": r["skill_match"],
    "Experience": r["experience_match"],
    "Education": r["education_match"],
    "Recommendation": r["recommendation"],
} for i, r in enumerate(results, 1)]

pd.DataFrame(rows)

In [ ]:
for r in results:
    print("=" * 70)
    print(r["name"])
    print(f"Overall score : {r['overall_score']:.2f}")
    print(f"Semantic      : {r['semantic_similarity']:.2f}%")
    print(f"Skill match   : {r['skill_match']:.2f}%")
    print(f"Experience    : {r['experience_match']:.2f}%")
    print(f"Education     : {r['education_match']:.2f}%")
    print("Matched skills:", ", ".join(r["matched_skills"]) or "-")
    print("Missing skills:", ", ".join(r["missing_skills"]) or "-")
    print("Recommendation:", r["recommendation"])

## Scoring Design

- Skill Match: **30%**
- Experience Match: **25%**
- Education Match: **15%**
- Semantic Similarity: **30%**

Recommendation thresholds:
- **≥ 80:** SHORTLIST
- **65–79.99:** REVIEW
- **< 65:** REJECT

The system is intended as **decision support**, not autonomous hiring. Recruiters should review evidence before making a final decision.